# Telco Customer Churn — Baseline Classification Models

**Dataset:** IBM sample Telco Customer Churn (Kaggle: `blastchar/telco-customer-churn`)  
**Models:** Decision Tree baseline and Random Forest comparison  
**Goal:** Predict `Churn` (Yes=1, No=0) using customer attributes.

This notebook runs preprocessing, training, and evaluation with actual held-out results. It does not yet implement Streamlit or drift monitoring.

**Run instructions:** Download the CSV following `data/README.md` and save it as `data/WA_Fn-UseC_-Telco-Customer-Churn.csv` from the repository root. Then run all notebook cells. In Google Colab, upload the CSV using the Files pane; the notebook also searches `/content` and its `data/` folder.


In [1]:
import pandas as pd
import numpy as np
import sklearn
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

RANDOM_STATE = 42
file_name = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
search_paths = [
    Path('data') / file_name,                 # launched from repository root / Colab
    Path('../data') / file_name,              # launched from notebooks/ locally
    Path(file_name),                          # uploaded directly to Colab / working dir
    Path('/content/data') / file_name,        # Colab data folder
    Path('/content') / file_name,             # Colab direct upload
]
path = next((p for p in search_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError(
        f'Could not find {file_name}. Download it from the Kaggle link in data/README.md '
        'and place it in data/ (or upload it to the Colab Files pane).'
    )
df = pd.read_csv(path)
print('Python packages: pandas', pd.__version__, '| scikit-learn', sklearn.__version__)
print('Original data shape:', df.shape)
print('Churn distribution (percent):')
print((df['Churn'].value_counts(normalize=True) * 100).round(2).to_string())

Python packages: pandas 2.2.3 | scikit-learn 1.8.0
Original data shape: (7043, 21)
Churn distribution (percent):
Churn
No     73.46
Yes    26.54


## 1. Cleaning and defining predictors

`TotalCharges` is stored as text and has blank entries. Convert it to numeric and impute missing values with 0 (these rows correspond to zero-month tenure in this dataset). Drop `customerID` as a predictor, but retain the original CSV unchanged. Treat `SeniorCitizen` as categorical even though coded with integers.

In [2]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_charges = int(df['TotalCharges'].isna().sum())
print('Missing TotalCharges:', missing_charges)
print('All missing TotalCharges have tenure=0:', bool((df.loc[df['TotalCharges'].isna(), 'tenure'] == 0).all()))
df.loc[df['TotalCharges'].isna() & df['tenure'].eq(0), 'TotalCharges'] = 0
assert df['TotalCharges'].notna().all()
assert df['customerID'].is_unique
assert set(df['Churn'].unique()) == {'Yes', 'No'}
y = df['Churn'].map({'No': 0, 'Yes': 1})
X = df.drop(columns=['Churn', 'customerID']).copy()
X['SeniorCitizen'] = X['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
print('Predictor shape:', X.shape)
print('Remaining missing predictor values:', int(X.isna().sum().sum()))

Missing TotalCharges: 11
All missing TotalCharges have tenure=0: True
Predictor shape: (7043, 19)
Remaining missing predictor values: 0


## 2. Stratified train/test split and leakage-safe preprocessing

Use 80% training / 20% held-out testing. Categorical one-hot encoding and numeric imputation are learned from the **training set only** via a scikit-learn pipeline. The same split is used for both models. No hyperparameter tuning or test-based model selection is performed here.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
numeric_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.select_dtypes(exclude=np.number).columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ('numeric', SimpleImputer(strategy='median'), numeric_columns),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
])
print('Training examples:', len(X_train), '| Test examples:', len(X_test))
print('Training churn rate:', round(y_train.mean(), 4), '| Test churn rate:', round(y_test.mean(), 4))
print('Numeric fields:', numeric_columns)
print('Categorical fields:', categorical_columns)

Training examples: 5634 | Test examples: 1409
Training churn rate: 0.2654 | Test churn rate: 0.2654
Numeric fields: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical fields: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 3. Fit baseline and comparison models

The Decision Tree is the primary baseline. The Random Forest is a simple comparison, **not** a model that learns continuously from incoming predictions. Both remain unchanged unless retrained. Fixed seeds ensure reproducible runs.

In [4]:
models = {
    'Decision Tree (baseline)': Pipeline([
        ('prep', preprocessor),
        ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    'Random Forest (comparison)': Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))
    ])
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'Trained: {name}')

Trained: Decision Tree (baseline)


Trained: Random Forest (comparison)


## 4. Evaluation on the held-out test set

**Primary metrics:** Recall and F1 for the positive (churn=1) class. Accuracy alone can be misleading because churn is less common. **Additional metrics:** precision and ROC-AUC. Report the confusion matrix (rows: actual, columns: predicted) for transparency.

Metrics are computed using the default **0.50 probability threshold**, not a threshold selected on the test set.

In [5]:
results = []
for name, model in models.items():
    pred = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision (churn)': precision_score(y_test, pred, zero_division=0),
        'Recall (churn)': recall_score(y_test, pred, zero_division=0),
        'F1 (churn)': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, probability)
    })
metrics = pd.DataFrame(results).set_index('Model')
print('Held-out test-set performance:')
print(metrics.round(4).to_string())
for name, model in models.items():
    pred = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0,1]).ravel()
    print(f'\n{name} — Confusion matrix')
    print('                     Pred No    Pred Yes')
    print(f'Actual No (0)       {tn:7d}    {fp:7d}')
    print(f'Actual Yes (1)      {fn:7d}    {tp:7d}')
print('\nMajority-class (predict No for everyone) accuracy:', round((y_test==0).mean(), 4))

Held-out test-set performance:
                            Accuracy  Precision (churn)  Recall (churn)  F1 (churn)  ROC-AUC
Model                                                                                       
Decision Tree (baseline)      0.7168             0.4675          0.4813      0.4743   0.6414
Random Forest (comparison)    0.7850             0.6203          0.4893      0.5471   0.8200

Decision Tree (baseline) — Confusion matrix
                     Pred No    Pred Yes
Actual No (0)           830        205
Actual Yes (1)          194        180



Random Forest (comparison) — Confusion matrix
                     Pred No    Pred Yes
Actual No (0)           923        112
Actual Yes (1)          191        183

Majority-class (predict No for everyone) accuracy: 0.7346


## 5. Scope and limitations

- The dataset is a static snapshot. This baseline does **not** represent online learning or a live dashboard.
- Both models use default class thresholds and largely default settings, so neither is claimed to be tuned or optimal.
- The test set supports a one-time evaluation. For later simulations of incoming batches and drift, use a distinct held-out simulation partition or redesign the split; do not repeatedly tune or retrain on the same final test set.
- Customer churn outcomes would normally arrive after a delay. Performance metrics require known ground-truth labels; monitoring input distributions and prediction rates does not.
- `TotalCharges` and tenure are features in this snapshot, not a timestamped sequence of historical observations.
